<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/25_flash_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00


In [2]:
import torch
import math

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B, S, D = Q.shape
    result = torch.zeros_like(Q, device=Q.device)
    # Q blocks
    for i in range(0, S, block_size):
      Qb = Q[:, i:i+block_size]
      Sb = Qb.shape[1]
      row_sum = torch.zeros((B, Sb, 1), device=Qb.device, dtype=torch.float32)
      row_max = torch.full((B, Sb, 1), float('-inf'), device=Qb.device, dtype=torch.float32)
      acc = torch.zeros_like(Qb)
      for j in range(0, S, block_size):
        kb = K[:, j:j+block_size]
        vb = V[:, j:j+block_size]
        scores = torch.bmm(Qb, kb.transpose(1, 2)) / math.sqrt(D)
        block_max = scores.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(block_max, row_max)
        correction = torch.exp(row_max - new_max)
        exp_scores = torch.exp(scores - new_max)
        acc = acc * correction + torch.bmm(exp_scores, vb)
        row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)
        row_max = new_max
      result[:, i:i+block_size] = acc / row_sum
    return result

In [6]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [7]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (26.7ms)
  ✅ [2/4] Non-aligned block size (2.6ms)
  ✅ [3/4] Block size invariant (2.6ms)
  ✅ [4/4] Gradient flow (40.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (71.9ms total)
  Progress saved. Run status() to see your dashboard.

